In [0]:
SILVER_PATH = "/Volumes/workspace/default/insure_data/silver"
GOLD_PATH = "/Volumes/workspace/default/insure_data/gold"
 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_agent = spark.read.format("delta").load(f"{SILVER_PATH}/agent")
silver_agent_policy = spark.read.format("delta").load(f"{SILVER_PATH}/agent_policy")
silver_policy = spark.read.format("delta").load(f"{SILVER_PATH}/policy")
silver_product = spark.read.format("delta").load(f"{SILVER_PATH}/product_master")

In [0]:
sales_df = (
    silver_agent
    .join(silver_agent_policy, "agent_no", "inner")
    .join(silver_policy, "policy_no", "inner")
    .join(silver_product, "product_id", "inner")
)

In [0]:
product_sales_city = (
    sales_df
    .groupBy(
        "city",
        "product_id",
        "product_name",
        "category"
    )
    .agg(
        F.countDistinct("policy_no").alias("policies_sold")
    )
)

In [0]:
window_spec = Window.partitionBy("city") \
                    .orderBy(F.desc("policies_sold"))

gold_product_sales_city = (
    product_sales_city
    .withColumn("rank", F.dense_rank().over(window_spec))
    .filter(F.col("rank") == 1)
    .drop("rank")
)

In [0]:
print("Total Records :", gold_product_sales_city.count())

display(
    gold_product_sales_city.orderBy("city")
)

In [0]:
#--to save in delta table format 
# Drop duplicate columns before saving
sales_df_clean = sales_df.select(
    "product_id", "policy_no", "agent_no", "source_agent_no",
    "first_name", "middle_name", "last_name", "dob", "gender",
    "mobile_no", "email", "joining_date", "termination_date",
    "designation", "branch_code", "branch_name", "city", "state",
    "country", "agent_status", "agent_full_name", "agent_location",
    "split_percentage", "source_policy_no", "policy_status",
    "sum_assured", "product_name", "category"
)

(
    sales_df_clean.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(f"{GOLD_PATH}/Sales_data")
)

print("Gold Delta table created successfully.")

'''
#--to save in CSV format
CSV_PATH = "/Volumes/workspace/default/insure_data/gold/sales_data_BI/"
(
    sales_df_clean.coalesce(1)      # Creates a single CSV file
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(CSV_PATH)
)
print("Gold CSV created successfully.")
'''



In [0]:
# rename csv

files = dbutils.fs.ls("/Volumes/workspace/default/insure_data/gold/sales_data_BI/")

for file in files:
    if file.name.endswith(".csv"):
        dbutils.fs.mv(
            file.path,
            "/Volumes/workspace/default/insure_data/gold/Sales_By_Area.csv"
        )